# Datamine RNLM Process Example

This notebook demonstrates how to configure and run the **`rnlm`** process wrapper in `dmstudio`.

## Process Description

## RNLM

R - mode non linear mapping groups together elements using the angular distances calculated from the Pearsson correlation coefficient matrix.

Note the difference with [Q mode non linear mapping](<qnlm.md>) which clusters samples on the basis of composition.

A two dimensional view of the element clusters, or the scores **NLM-X** versus **NLM-Y** , is calculated to represent them as they would appear in multi-dimensional space on the basis of element similarity between pairs of samples. All input data is standardized prior to the calculation of the correlation matrix and the linear mapping output scores are normalized prior to plotting.

R - mode non linear mapping, when compared with R - mode factor analysis, is more sophisticated and does not distort or sub-divide the element or parameter clusters. Non linear mapping will give more separable clusters than other hierarchical techniques such as R - mode cluster analysis.

**Note** : There is a restriction of 2000 samples.

### File Handling

The input file &(**IN**) must have a separate identifier field (* **SAMPID**). The output file &(**SCORES**) contains three parameters, **FIELD** the field identifier, **NLM-X** and **NLM-Y** the output scores for plotting the multi-dimensional inter-element distances as a two dimensional flat plane. Results can be displayed using **[QUIG](<quig.md>)** , **[PLOTDA](<plotda.md>)** or [PLOTAN](<plotan.md>).

### Iteration Procedure

In order to present a two dimensional view of multi-dimensional space with minimum distortion of the sample clusters, the calculated mapping error has to be minimized by an iterative method (steepest descent). This is controlled by the @**CONVLIM** parameter, that is the minimum difference allowed in the mapping error between iterations, @**MAXIT** , the maximum number of iterations permitted and the @**MAGIC** parameter which specifies the stepping function used for each iteration. If the stepping function @**MAGIC** is decreased, the number of iterations is increased with an obvious time penalty on the length of the calculation. The value used must be taken into account when there are a large number of samples. However the results can be more stable.

### Input Files:

* **in** (Undefined):
  Input file.
  Required=Yes

### Output Files:

* **scores** (Undefined):
  Optional output file for principal component scores.
  Required=No

### Fields:

* **sampid** (Any : IN):
  Field containing sample identification
  Default=Undefined
  Required=Yes

* **fields** (Undefined : Undefined):
  Fields to be used. No fields specified means all.
  Default=Undefined
  Required=No

### Parameters:

* **convlim**:
  Convergence limit.
  Range=Undefined
  Values=Undefined
  Default=0.0001
  Required=No

* **magic**:
  Convergence [magic] factor.
  Range=Undefined
  Values=Undefined
  Default=0.35
  Required=No

* **maxit**:
  Maximum number of iterations .
  Range=Undefined
  Values=Undefined
  Default=100
  Required=No

* **print**:

* **Options** ((0): Print configuration results to screen.):
  Range=0,1
  Values=0,1
  Default=0
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('rnlm')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically (4 levels up from subfolders)
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute rnlm
print("Running rnlm...")
dm_cmd.rnlm(
    in_i='t_assays',  # required
    scores_o=['t_rnlm_out'],  # required
    sampid_f='optional',  # required
    fields_f=['AU'],  # required
    # convlim_p=0.0001,  # optional
    # magic_p=0.35,  # optional
    # maxit_p=100,  # optional
    # print_p=0,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("rnlm execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_rnlm_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")